# 01. Khám phá Dữ liệu (EDA) & Tiền xử lý (Data Preprocessing)
**Học phần:** Học máy (Machine Learning) - HUCE  
**Bài toán:** Dự đoán bảo trì nhà máy thông minh (Predictive Maintenance)  
**Mục tiêu:** Thu thập dữ liệu, kiểm tra tính toàn vẹn (missing, duplicates, outliers), phân tích phân phối, tương quan đặc trưng và thiết kế chiến lược tiền xử lý không rò rỉ dữ liệu (No Data Leakage).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình trực quan hóa
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Tải và kiểm tra tổng quan dữ liệu

In [ ]:
data_path = os.path.join("..", "data", "ai4i+2020+predictive+maintenance+dataset", "ai4i2020.csv")
df = pd.read_csv(data_path)
print(f"Kích thước dữ liệu: {df.shape}")
df.head()

## 2. Kiểm tra chất lượng dữ liệu (Missing Values, Duplicates, Data Types)

In [ ]:
print("Thông tin kiểu dữ liệu và giá trị thiếu:")
df.info()
print(f"\nSố dòng trùng lặp: {df.duplicated().sum()}")

## 3. Phân tích phân phối biến mục tiêu (Mất cân bằng dữ liệu)
Phân tích tỷ lệ nhãn `Machine failure` để định hướng chiến lược đánh giá (dùng F1, PR-AUC thay vì chỉ Accuracy).

In [ ]:
target_counts = df["Machine failure"].value_counts()
target_pct = df["Machine failure"].value_counts(normalize=True) * 100
print(pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct}))

## 4. Phân tích tương quan & Phân phối đặc trưng vật lý

In [ ]:
features_num = [
    "Air temperature [K]", "Process temperature [K]", 
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"
]

plt.figure(figsize=(8, 6))
sns.heatmap(df[features_num].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Ma trận tương quan giữa các đặc trưng số")
plt.show()

## 5. Kết luận phân tích EDA & Định hướng tiền xử lý trong Pipeline
- **Loại bỏ cột định danh:** `UDI`, `Product ID` không mang giá trị quy luật vật lý.
- **Chống rò rỉ dữ liệu:** Tuyệt đối không đưa các cờ lỗi thành phần (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) vào ma trận đặc trưng $X$.
- **Mã hóa:** Áp dụng `OneHotEncoder` cho biến hạng mục `Type`.
- **Chuẩn hóa:** Áp dụng `StandardScaler` cho các đặc trưng số bên trong `ColumnTransformer`.